# PyTorch 101

Quick notes before you start:
- `torch.Tensor` is the main data type in PyTorch, similar to a numpy array.
- Cells are built step by step: tensors, tensor math, autograd, then a small model.
- Run the cells in order, from top to bottom.

In [ ]:
import torch

## Tensors

- A tensor is a table of numbers (1D, 2D, 3D, ...).
- `shape` = size of each dimension.
- `dtype` = type of numbers inside (int, float, ...).
- `ndim` = number of dimensions.

In [ ]:
token_ids = torch.tensor([
                [10, 42, 7, 3],
                [88, 15, 0, 3],
                [90, 2, 7, 63]
            ])


print(token_ids.shape)   # torch.Size([3, 4])
print(token_ids.dtype)   # torch.int64
print(token_ids.ndim)    # 2

## Creating tensors

Common ways to make tensors:
- `torch.zeros(rows, cols)` → all values 0
- `torch.ones(rows, cols)` → all values 1
- `torch.randn(rows, cols)` → random values
- `torch.arange(start, end)` → a range of numbers

In [ ]:
zeros = torch.zeros(3, 5)
ones  = torch.ones(2, 3)

rand_tensor  = torch.randn(2, 3)

steps = torch.arange(0, 6)

## Tensor math

Math between tensors happens element by element (same position in both tensors).

In [ ]:
# Element-wise

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(a + b)

tensor([5., 7., 9.])


In [ ]:
# Element-wise

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(a * b)

tensor([ 4., 10., 18.])


## Matrix multiplication

- `@` is the matrix multiplication operator.
- For `x1 @ x2`, the last dimension of `x1` must match the first dimension of `x2`.
- Example: (2, 3) @ (3, 4) → (2, 4)

In [ ]:
x1 = torch.randn(2, 3)
x2 = torch.randn(3, 4)

In [ ]:
print(x1)
print(x2)

y = x1 @ x2
print(y.shape)

## Reshaping tensors

- `view()` changes the shape without changing the data.
- The total number of elements must stay the same before and after.

In [ ]:
x = torch.randn(2, 4, 6) # (B=2, T=4, C=6)

# The total number of elements must stay the same (2*4*6 = 48).
flat = x.view(2, 24)         # (2, 24)  — merged T and C
back = flat.view(2, 4, 6)    # (2, 4, 6) — split them again


torch.Size([2, 4, 6])

## Transpose

- `transpose(dim1, dim2)` swaps two dimensions.
- Shape changes, but the values stay the same, just in different positions.

In [ ]:
x = torch.randn(2, 4, 6) # (B=2, T=4, C=6)

x_transp = x.transpose(1, 2)

# (2, 4, 6)
#     ↘ ↙
# (2, 6, 4)

print( x_transp.shape )

## reshape() vs view()

- After `transpose()`, the data in memory is no longer in simple order.
- `view()` needs the memory to be in order ("contiguous").
- `reshape()` works anyway, it copies the data if needed.

In [ ]:
x = torch.randn(2, 4, 6) # (B=2, T=4, C=6)

y = x.transpose(1, 2).reshape(2, 24)
y.shape

torch.Size([2, 24])

To use `view()` after `transpose()`, call `.contiguous()` first. This copies the data into normal order, so `view()` works.

In [ ]:
x = torch.randn(2, 4, 6) # (B=2, T=4, C=6)

y = x.transpose(1, 2).contiguous().view(2, 24)

## Using the GPU

- `.to("cuda:0")` moves a tensor to the GPU.
- This only works if your machine has a GPU with CUDA. If not, you will see an error, that is normal, just skip it.

In [ ]:
import torch

a = torch.tensor([1.0, 2.0, 3.0]).to("cuda:0")
b = torch.tensor([4.0, 5.0, 6.0]).to("cuda:0")

print( a + b )

tensor([5., 7., 9.], device='cuda:0')


## Autograd (automatic gradients)

- `requires_grad=True` tells PyTorch to track operations on this tensor.
- `.backward()` computes the gradient, how much the result changes if `w` changes a little.

In [ ]:
w = torch.tensor([2.0], requires_grad=True)
x = torch.tensor([3.0])

loss = (w * x - 1) ** 2

loss.backward()

Picture of what just happened:

In [ ]:
# x
# │
# ▼
# weight * x + bias
# │
# ▼
# prediction
# │
# ▼
# compare with target
# │
# ▼
# loss

## Building a small model

`nn.Module` is the base class for models in PyTorch. Every model needs:
- `__init__` → define the parts (weights, layers)
- `forward` → define how input turns into output

In [ ]:
import torch
import torch.nn as nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()

        # self.w = torch.tensor([2.0], requires_grad=True)
        # self.b = torch.tensor([0.0], requires_grad=True)

        self.w = nn.Parameter(torch.tensor([2.0]))
        self.b = nn.Parameter(torch.tensor([0.0]))


    def forward(self, x):
        # return prediction
        return (self.w * x) + self.b

- `nn.Parameter` marks a tensor as something the model should learn (PyTorch tracks its gradient automatically).
- Compare it to the commented-out lines above, written by hand with `requires_grad=True`.

Now create the model and run one prediction:

In [ ]:
my_model = TinyModel()

print("w:", my_model.w)
print("b:", my_model.b)
target = 18
learning_rate = 0.0002

my_input = torch.tensor([3.0])

prediction = my_model(my_input) # my_model.forward(x = my_input)

w: Parameter containing:
tensor([2.], requires_grad=True)
b: Parameter containing:
tensor([0.], requires_grad=True)


## Loss

Loss shows how far the prediction is from the target. Smaller loss means a better prediction.

In [ ]:
loss = (prediction - target) ** 2
print(loss)

tensor([144.], grad_fn=<PowBackward0>)


Call `.backward()` to compute the gradients of the loss with respect to `w` and `b`.

In [ ]:
loss.backward()

`.backward()` only computes gradients. It does NOT change `w` or `b` yet, check below.

In [ ]:
print("w:", my_model.w)
print("b:", my_model.b)

w: Parameter containing:
tensor([2.], requires_grad=True)
b: Parameter containing:
tensor([0.], requires_grad=True)


Gradients are stored in `.grad`, separate from the parameter value.

In [ ]:
print("w:", my_model.w.grad)
print("b:", my_model.b.grad)

w: tensor([-72.])
b: tensor([-24.])


What the gradients mean, and how we use them to update `w` and `b`:

In [ ]:
# w.grad = d(loss) / d(w)
# b.grad = d(loss) / d(b)

In [ ]:
# w (new_value) = w (current) - ( learning_rate * w.grad )

Full picture, from input to updated weights:

In [ ]:
#         FORWARD

# x
# │
# ├──► w*x + b
# │
# │
# ▼
# prediction
# │
# ▼
# loss



#         BACKWARD

# loss
# │
# ▼
# calculate gradients
# │
# ├──► w.grad
# └──► b.grad



#         UPDATE

# w = w - learning_rate * w.grad
# b = b - learning_rate * b.grad




#       Clear old gradients


# model.w.grad.zero_()
# model.b.grad.zero_()

## Training loop

Same steps as above (forward → loss → backward → update → clear gradients), repeated many times. Watch how `loss` gets smaller and `w` gets closer to the target.

In [ ]:
import torch
import torch.nn as nn


class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.w = nn.Parameter(torch.tensor([2.0]))
        self.b = nn.Parameter(torch.tensor([0.0]))

    def forward(self, x):
        return self.w * x + self.b


model = TinyModel()

x = torch.tensor([3.0])

target = torch.tensor([1.0])
learning_rate = 0.01


for step in range(15):

    # -------------------------
    # 1. Forward
    # -------------------------
    prediction = model(x)

    # -------------------------
    # 2. Loss
    # -------------------------
    loss = (prediction - target) ** 2

    # -------------------------
    # 3. Backward
    # -------------------------
    loss.backward()

    print(
        f"step={step}",
        f"prediction={prediction.item():.2f}",
        f"loss={loss.item():.2f}",
        f"w.grad={model.w.grad.item():.2f}",
        "=====",
        f"curent w = {model.w.item():.2f}",
    )

    # -------------------------
    # 4. Update parameters
    # -------------------------
    with torch.no_grad():
        model.w -= learning_rate * model.w.grad
        model.b -= learning_rate * model.b.grad

    # -------------------------
    # 5. Clear old gradients
    # -------------------------
    model.w.grad.zero_()
    model.b.grad.zero_()

step=0 prediction=6.00 loss=25.00 w.grad=30.00 ===== curent w = 2.00
step=1 prediction=5.00 loss=16.00 w.grad=24.00 ===== curent w = 1.70
step=2 prediction=4.20 loss=10.24 w.grad=19.20 ===== curent w = 1.46
step=3 prediction=3.56 loss=6.55 w.grad=15.36 ===== curent w = 1.27
step=4 prediction=3.05 loss=4.19 w.grad=12.29 ===== curent w = 1.11
step=5 prediction=2.64 loss=2.68 w.grad=9.83 ===== curent w = 0.99
step=6 prediction=2.31 loss=1.72 w.grad=7.86 ===== curent w = 0.89
step=7 prediction=2.05 loss=1.10 w.grad=6.29 ===== curent w = 0.81
step=8 prediction=1.84 loss=0.70 w.grad=5.03 ===== curent w = 0.75
step=9 prediction=1.67 loss=0.45 w.grad=4.03 ===== curent w = 0.70
step=10 prediction=1.54 loss=0.29 w.grad=3.22 ===== curent w = 0.66
step=11 prediction=1.43 loss=0.18 w.grad=2.58 ===== curent w = 0.63
step=12 prediction=1.34 loss=0.12 w.grad=2.06 ===== curent w = 0.60
step=13 prediction=1.27 loss=0.08 w.grad=1.65 ===== curent w = 0.58
step=14 prediction=1.22 loss=0.05 w.grad=1.32 ====

## Towards a language model

New topic: how a language model turns token ids into predictions.
- `B` = batch size (how many sequences)
- `T` = sequence length (how many tokens per sequence)

In [ ]:
# token_ids
# shape: (B, T)

# example:
# [[ 42,  18, 901, ... ],
#  [  7, 123,  55, ... ]]


In [ ]:

# token_ids
# shape: (B, T)

# example:
# [[ 42,  18, 901, ... ],
#  [  7, 123,  55, ... ]]


#         │
#         ▼

# ┌─────────────────────────────┐
# │       nn.Embedding          │
# │                             │
# │  24,000 possible tokens     │
# │        ↓                    │
# │  each token gets a vector   │
# │  of size 768                │
# └─────────────────────────────┘

#         │
#         │  (B, T)
#         │      ↓
#         │  (B, T, 768)
#         ▼

# x = embeddings
# shape: (B, T, 768)

# Each token is now represented by
# 768 floating-point numbers.

#         │
#         ▼

# ┌─────────────────────────────┐
# │       nn.LayerNorm          │
# │                             │
# │   normalize each token's    │
# │       768 features          │
# └─────────────────────────────┘

#         │
#         │ shape stays the same
#         │
#         ▼

# x
# shape: (B, T, 768)

#         │
#         ▼

# ┌─────────────────────────────┐
# │        nn.Linear            │
# │          "head"             │
# │                             │
# │       768 inputs            │
# │           ↓                 │
# │     24,000 outputs          │
# │                             │
# │ one score for every token   │
# │      in the vocabulary      │
# └─────────────────────────────┘

#         │
#         │ (B, T, 768)
#         │      ↓
#         │ (B, T, 24,000)
#         ▼

# logits
# shape: (B, T, 24,000)

# For EVERY position:
# 24,000 scores

#         │
#         ▼

# "Which vocabulary token
# should come next?"

In [ ]:
# token ID
#    ↓
# Embedding
#    ↓
# vector of 768 numbers
#    ↓
# LayerNorm
#    ↓
# cleaned/normalized vector
#    ↓
# Linear head
#    ↓
# 24,000 scores

Same pipeline, written as real PyTorch code:

In [ ]:
import torch
import torch.nn as nn

class CustomModel(nn.Module):

    def __init__(self, vocab_size:int, dim:int):
        super().__init__()

        # token ID
        #    ↓
        # vector of 768 numbers
        self.embedding = nn.Embedding(vocab_size, dim)

        # vector of 768 numbers
        #    ↓
        # LayerNorm
        self.norm = nn.LayerNorm(dim)

        # 768 numbers
        #    ↓
        # 24,000 vocabulary scores
        self.ff_head = nn.Linear(dim, vocab_size)

    def forward(self, token_ids):
        embed_output = self.embedding(token_ids)

        embed_output = self.norm(embed_output)

        logits = self.ff_head(embed_output)

        return logits

# ========================

# create
my_model = CustomModel(
    vocab_size=24_000,
    dim=768
)

token_ids = torch.tensor([
    [10, 25, 98, 16],
    [4,  77, 31, 9]
])

token_logits = my_model(token_ids)